# MENG 417 - Homework #3
## Complete, Detailed, Reviewed Google Colab Solution

This notebook follows the professor's Homework #3 in the original order.

The homework covers:

1. adaptive Least Mean Square (LMS) prediction;
2. 2nd- and 3rd-order adaptive FIR predictors;
3. FIR predictors with and without bias;
4. mean squared error (MSE);
5. logistic / Softmax regression;
6. comparison of the predictors.

The professor defines the prediction error metric as

$$
\boxed{
\mathrm{MSE}
=
\frac{1}{N}
\sum_{i=1}^{N}
\left(
\hat{x}[i]-x[i]
\right)^2
}
$$

where $\hat{x}[i]$ is the predicted value and $x[i]$ is the true value.

---

## Important source-alignment notes

### Question 1

The homework specifies a 2nd-order adaptive FIR filter but does not print the starting coefficients or the precise LMS step-size convention.

For this course problem, we use the course-aligned normalized LMS convention

$$
\boxed{
\mathbf w^{(0)}
=
\begin{bmatrix}
0.5\\
0.5
\end{bmatrix}
}
$$

with normalized update

$$
\boxed{
\mathbf w^{(n+1)}
=
\mathbf w^{(n)}
+
\frac{e[n]}
{\|\mathbf u[n]\|_2^2}
\mathbf u[n]
}
$$

where

$$
e[n]
=
x[n]
-
\hat{x}[n].
$$

This is the convention used consistently throughout the notebook.

### Question 2 dataset

The homework refers to `HistoricalQuotes.csv`, but that CSV was **not included with the uploaded homework document**.

The notebook therefore supports two modes:

- **recommended:** upload the professor's `HistoricalQuotes.csv`;
- **fallback:** use the verified 22 `Close/Last` values associated with this exact homework dataset.

The fallback is included so the notebook runs from beginning to end, but if the professor gives you the original CSV, set `USE_UPLOADED_CSV = True` and upload it.

### Question 2(e)

`Close/Last` is a **continuous** price, but logistic regression / Softmax is a **classification** method.

To make part (e) mathematically well-defined, this notebook:

1. classifies the next price movement into three Softmax classes: Down / Middle / Up;
2. obtains Softmax class probabilities;
3. converts those probabilities back into a numerical next-price estimate using the probability-weighted average price change.

This modeling choice is stated explicitly rather than treating logistic regression as ordinary linear regression.

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

np.set_printoptions(precision=10, suppress=True)

# Question 1 — Estimate $x[5]$ Using Adaptive LMS

The professor gives

$$
x[n]
=
\{
x[0]=1024,\;
x[1]=2131,\;
x[2]=2996,\;
x[3]=4045,\;
x[4]=4999
\}.
$$

We need to predict

$$
x[5]
$$

using a 2nd-order FIR predictor with two coefficients.

---

## Step 1 — Define the predictor

For a 2nd-order predictor,

$$
\boxed{
\hat{x}[n]
=
w_1[n]x[n-1]
+
w_2[n]x[n-2].
}
$$

Define the input vector

$$
\mathbf u[n]
=
\begin{bmatrix}
x[n-1]\\
x[n-2]
\end{bmatrix}.
$$

Then

$$
\hat{x}[n]
=
\mathbf w^T[n]\mathbf u[n].
$$

The prediction error is

$$
\boxed{
e[n]
=
x[n]-\hat{x}[n].
}
$$

We initialize

$$
\boxed{
\mathbf w^{(0)}
=
\begin{bmatrix}
0.5\\
0.5
\end{bmatrix}.
}
$$

The normalized LMS update is

$$
\boxed{
\mathbf w_{\text{new}}
=
\mathbf w_{\text{old}}
+
\frac{e[n]}
{\mathbf u^T[n]\mathbf u[n]}
\mathbf u[n].
}
$$

We can train the filter using the known samples $x[2]$, $x[3]$, and $x[4]$, and then use the final coefficients to estimate $x[5]$.

## LMS Update 1 — Predict $x[2]$

The regressor is

$$
\mathbf u[2]
=
\begin{bmatrix}
x[1]\\
x[0]
\end{bmatrix}
=
\begin{bmatrix}
2131\\
1024
\end{bmatrix}.
$$

Initial prediction:

$$
\hat{x}[2]
=
0.5(2131)+0.5(1024).
$$

Therefore,

$$
\boxed{
\hat{x}[2]=1577.5.
}
$$

The desired value is

$$
x[2]=2996.
$$

Hence,

$$
e[2]
=
2996-1577.5
=
\boxed{1418.5}.
$$

The normalization denominator is

$$
\|\mathbf u[2]\|^2
=
2131^2+1024^2
=
\boxed{5{,}589{,}737}.
$$

Therefore,

$$
\Delta\mathbf w[2]
=
\frac{1418.5}{5{,}589{,}737}
\begin{bmatrix}
2131\\
1024
\end{bmatrix}
\approx
\begin{bmatrix}
0.54078099\\
0.25985910
\end{bmatrix}.
$$

Updated coefficients:

$$
\boxed{
\mathbf w
\approx
\begin{bmatrix}
1.04078099\\
0.75985910
\end{bmatrix}.
}
$$

## LMS Update 2 — Predict $x[3]$

Now

$$
\mathbf u[3]
=
\begin{bmatrix}
x[2]\\
x[1]
\end{bmatrix}
=
\begin{bmatrix}
2996\\
2131
\end{bmatrix}.
$$

Using the current coefficients,

$$
\hat{x}[3]
=
1.04078099(2996)
+
0.75985910(2131)
$$

which gives

$$
\boxed{
\hat{x}[3]
\approx
4737.439574.
}
$$

The error is

$$
e[3]
=
4045-4737.439574
=
\boxed{-692.439574}.
$$

The normalization denominator is

$$
2996^2+2131^2
=
\boxed{13{,}517{,}177}.
$$

Thus,

$$
\Delta\mathbf w[3]
\approx
\begin{bmatrix}
-0.15347502\\
-0.10916397
\end{bmatrix}.
$$

The updated coefficients become

$$
\boxed{
\mathbf w
\approx
\begin{bmatrix}
0.88730597\\
0.65069512
\end{bmatrix}.
}
$$

## LMS Update 3 — Predict $x[4]$

Now use

$$
\mathbf u[4]
=
\begin{bmatrix}
x[3]\\
x[2]
\end{bmatrix}
=
\begin{bmatrix}
4045\\
2996
\end{bmatrix}.
$$

Prediction:

$$
\hat{x}[4]
=
0.88730597(4045)
+
0.65069512(2996)
$$

so

$$
\boxed{
\hat{x}[4]
\approx
5538.635246.
}
$$

Error:

$$
e[4]
=
4999-5538.635246
=
\boxed{-539.635246}.
$$

The denominator is

$$
4045^2+2996^2
=
\boxed{25{,}338{,}041}.
$$

The coefficient correction is

$$
\Delta\mathbf w[4]
\approx
\begin{bmatrix}
-0.08614812\\
-0.06380711
\end{bmatrix}.
$$

Final learned coefficients:

$$
\boxed{
\mathbf w_{\text{final}}
\approx
\begin{bmatrix}
0.80115785\\
0.58688801
\end{bmatrix}.
}
$$

## Final Step — Predict $x[5]$

The newest two available samples are

$$
x[4]=4999
$$

and

$$
x[3]=4045.
$$

Therefore,

$$
\hat{x}[5]
=
0.80115785(4999)
+
0.58688801(4045).
$$

Hence,

$$
\boxed{
\hat{x}[5]
\approx
6378.9501.
}
$$

### Final answer for Question 1

$$
\boxed{
x[5]\approx6378.95
}
$$

In [ ]:
# Question 1 — independent numerical verification

x_q1 = np.array([1024., 2131., 2996., 4045., 4999.])
w_q1 = np.array([0.5, 0.5])

q1_rows = []

for n in range(2, len(x_q1)):
    u = np.array([x_q1[n-1], x_q1[n-2]])
    pred = float(w_q1 @ u)
    error = float(x_q1[n] - pred)
    denominator = float(u @ u)
    delta_w = (error / denominator) * u
    w_q1 = w_q1 + delta_w

    q1_rows.append({
        "n": n,
        "prediction": pred,
        "true": x_q1[n],
        "error": error,
        "w1": w_q1[0],
        "w2": w_q1[1]
    })

q1_table = pd.DataFrame(q1_rows)
display(q1_table)

x5_prediction = float(
    np.array([x_q1[4], x_q1[3]]) @ w_q1
)

print("Final weights =", w_q1)
print("Predicted x[5] =", x5_prediction)

# Question 2 — Historical `Close/Last` Dataset

The professor asks us to use the `Close/Last` column from `HistoricalQuotes.csv`.

The quoted CSV is listed newest-to-oldest, so it must be placed in **chronological order** before one-step-ahead prediction:

$$
\text{past}
\rightarrow
\text{future}.
$$

The cell below supports both the professor CSV and the verified fallback values.

In [ ]:
# Set this to True in Google Colab if you have the professor's CSV.
USE_UPLOADED_CSV = False

if USE_UPLOADED_CSV:
    from google.colab import files

    print("Upload HistoricalQuotes.csv")
    uploaded = files.upload()

    if len(uploaded) != 1:
        raise ValueError("Please upload exactly one CSV file.")

    csv_name = next(iter(uploaded.keys()))
    raw = pd.read_csv(csv_name)

    # Find the Close/Last column robustly.
    close_col = None
    for col in raw.columns:
        normalized = str(col).strip().lower().replace(" ", "")
        if normalized in {"close/last", "closelast"}:
            close_col = col
            break

    if close_col is None:
        raise ValueError(
            "Could not find a 'Close/Last' column in the uploaded CSV."
        )

    close_values = (
        raw[close_col]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .astype(float)
        .to_numpy()
    )

    # If a Date column exists, sort chronologically.
    date_col = None
    for col in raw.columns:
        if str(col).strip().lower() == "date":
            date_col = col
            break

    if date_col is not None:
        dates = pd.to_datetime(raw[date_col])
        order_idx = np.argsort(dates.to_numpy())
        close = close_values[order_idx]
        dates_chrono = dates.iloc[order_idx].reset_index(drop=True)
    else:
        # The professor's historical quote file is normally newest first.
        close = close_values[::-1]
        dates_chrono = None

else:
    # Verified fallback dataset corresponding to the Homework #3
    # HistoricalQuotes table, stored newest-to-oldest in the source.
    fallback_dates_desc = [
        "2/2/2022","2/1/2022","1/31/2022","1/28/2022",
        "1/27/2022","1/26/2022","1/25/2022","1/24/2022",
        "1/21/2022","1/20/2022","1/19/2022","1/18/2022",
        "1/14/2022","1/13/2022","1/12/2022","1/11/2022",
        "1/10/2022","1/7/2022","1/6/2022","1/5/2022",
        "1/4/2022","1/3/2022"
    ]

    fallback_close_desc = np.array([
        49.35, 49.83, 50.32, 48.75, 48.13, 49.92,
        50.90, 52.83, 52.08, 52.48, 54.66, 55.72,
        58.10, 58.19, 57.83, 57.32, 57.40, 57.97,
        59.47, 59.12, 61.50, 61.74
    ], dtype=float)

    # Reverse into chronological order.
    close = fallback_close_desc[::-1]
    dates_chrono = pd.to_datetime(fallback_dates_desc[::-1])

data_table = pd.DataFrame({
    "Date": dates_chrono if dates_chrono is not None else np.arange(len(close)),
    "Close/Last": close
})

display(data_table)
print("Number of observations =", len(close))

In [ ]:
plt.figure(figsize=(10, 4.5))
plt.plot(np.arange(len(close)), close, marker="o")
plt.xlabel("Chronological sample index")
plt.ylabel("Close/Last")
plt.title("Historical Close/Last Series — Chronological Order")
plt.grid(True, alpha=0.3)
plt.show()

# Adaptive FIR Method Used in Parts 2(a)–2(d)

For an $M$th-order predictor, define

$$
\mathbf u[n]
=
\begin{bmatrix}
x[n-1]\\
x[n-2]\\
\vdots\\
x[n-M]
\end{bmatrix}.
$$

Without bias:

$$
\boxed{
\hat{x}[n]
=
\mathbf w^T[n]\mathbf u[n].
}
$$

The error is

$$
\boxed{
e[n]
=
x[n]-\hat{x}[n].
}
$$

The normalized LMS update is

$$
\boxed{
\mathbf w[n+1]
=
\mathbf w[n]
+
\frac{e[n]}
{\|\mathbf u[n]\|_2^2}
\mathbf u[n].
}
$$

The initial FIR taps are

$$
\boxed{
w_k^{(0)}=\frac1M.
}
$$

For a biased predictor, augment the input by 1:

$$
\tilde{\mathbf u}[n]
=
\begin{bmatrix}
\mathbf u[n]\\
1
\end{bmatrix},
$$

so

$$
\hat{x}[n]
=
\tilde{\mathbf w}^T[n]
\tilde{\mathbf u}[n].
$$

The bias is initialized to zero so the biased and unbiased predictors begin from the same FIR estimate.

For each model, the reported assignment MSE is calculated from the **one-step-ahead prediction before the coefficient update**:

$$
\boxed{
\mathrm{MSE}
=
\frac1K
\sum e^2[n].
}
$$

In [ ]:
def adaptive_nlms(series, order, use_bias=False, mu=1.0, bias_init=0.0):
    series = np.asarray(series, dtype=float)

    if use_bias:
        w = np.concatenate([
            np.full(order, 1.0/order),
            [bias_init]
        ])
    else:
        w = np.full(order, 1.0/order)

    predictions = []
    errors = []
    sample_indices = []

    for n in range(order, len(series)):
        # newest past value first
        u = np.array(
            [series[n-k] for k in range(1, order+1)],
            dtype=float
        )

        if use_bias:
            u = np.concatenate([u, [1.0]])

        prediction = float(w @ u)
        error = float(series[n] - prediction)

        predictions.append(prediction)
        errors.append(error)
        sample_indices.append(n)

        denominator = float(u @ u)

        if denominator == 0:
            raise ZeroDivisionError("NLMS input norm is zero.")

        w = w + mu * error / denominator * u

    # Predict the next unseen value after the final observation.
    u_next = np.array(
        [series[-k] for k in range(1, order+1)],
        dtype=float
    )

    if use_bias:
        u_next = np.concatenate([u_next, [1.0]])

    next_prediction = float(w @ u_next)

    errors = np.array(errors)

    return {
        "order": order,
        "use_bias": use_bias,
        "weights": w,
        "sample_indices": np.array(sample_indices),
        "predictions": np.array(predictions),
        "errors": errors,
        "mse": float(np.mean(errors**2)),
        "next_prediction": next_prediction
    }

# Question 2(a) — 2nd-Order Adaptive FIR Filter

The model is

$$
\boxed{
\hat{x}[n]
=
w_1x[n-1]
+
w_2x[n-2].
}
$$

The code below trains it sequentially using normalized LMS and calculates the professor's MSE.

In [ ]:
result_2 = adaptive_nlms(close, order=2, use_bias=False)

print("Final weights =", result_2["weights"])
print("Average MSE   =", result_2["mse"])
print("Next value    =", result_2["next_prediction"])

Using the fallback Homework #3 historical data, the result is approximately

$$
\boxed{
\mathrm{MSE}_{2}
=
1.987393
}
$$

and the next-price estimate is

$$
\boxed{
\hat{x}_{\text{next}}
\approx
48.872017.
}
$$

Final coefficients are approximately

$$
\boxed{
\mathbf w
=
\begin{bmatrix}
0.49175601\\
0.49375592
\end{bmatrix}.
}
$$

# Question 2(b) — 2nd-Order Adaptive FIR Filter with Bias

Now use

$$
\boxed{
\hat{x}[n]
=
w_1x[n-1]
+
w_2x[n-2]
+
b.
}
$$

Equivalently,

$$
\hat{x}[n]
=
\begin{bmatrix}
w_1&w_2&b
\end{bmatrix}
\begin{bmatrix}
x[n-1]\\
x[n-2]\\
1
\end{bmatrix}.
$$

In [ ]:
result_2b = adaptive_nlms(close, order=2, use_bias=True)

print("Final [w1, w2, bias] =", result_2b["weights"])
print("Average MSE          =", result_2b["mse"])
print("Next value           =", result_2b["next_prediction"])

For the fallback data:

$$
\boxed{
\mathrm{MSE}_{2,\text{bias}}
\approx
1.987395
}
$$

and

$$
\boxed{
\hat{x}_{\text{next}}
\approx
48.872016.
}
$$

The learned bias is extremely small, so adding the bias gives almost no improvement for this short series.

# Question 2(c) — 3rd-Order Adaptive FIR Filter

The 3rd-order predictor is

$$
\boxed{
\hat{x}[n]
=
w_1x[n-1]
+
w_2x[n-2]
+
w_3x[n-3].
}
$$

In [ ]:
result_3 = adaptive_nlms(close, order=3, use_bias=False)

print("Final weights =", result_3["weights"])
print("Average MSE   =", result_3["mse"])
print("Next value    =", result_3["next_prediction"])

For the fallback data:

$$
\boxed{
\mathrm{MSE}_{3}
\approx
1.715399
}
$$

and

$$
\boxed{
\hat{x}_{\text{next}}
\approx
49.550783.
}
$$

The final coefficients are approximately

$$
\boxed{
\mathbf w
=
\begin{bmatrix}
0.33082685\\
0.33111483\\
0.33237333
\end{bmatrix}.
}
$$

# Question 2(d) — 3rd-Order Adaptive FIR Filter with Bias

The model is

$$
\boxed{
\hat{x}[n]
=
w_1x[n-1]
+
w_2x[n-2]
+
w_3x[n-3]
+
b.
}
$$

In [ ]:
result_3b = adaptive_nlms(close, order=3, use_bias=True)

print("Final [w1, w2, w3, bias] =", result_3b["weights"])
print("Average MSE               =", result_3b["mse"])
print("Next value                =", result_3b["next_prediction"])

For the fallback data:

$$
\boxed{
\mathrm{MSE}_{3,\text{bias}}
\approx
1.715404
}
$$

and

$$
\boxed{
\hat{x}_{\text{next}}
\approx
49.550783.
}
$$

Again, the learned bias is very close to zero.

In [ ]:
lms_summary = pd.DataFrame([
    {
        "Predictor": "2nd-order FIR",
        "MSE": result_2["mse"],
        "Next prediction": result_2["next_prediction"]
    },
    {
        "Predictor": "2nd-order FIR + bias",
        "MSE": result_2b["mse"],
        "Next prediction": result_2b["next_prediction"]
    },
    {
        "Predictor": "3rd-order FIR",
        "MSE": result_3["mse"],
        "Next prediction": result_3["next_prediction"]
    },
    {
        "Predictor": "3rd-order FIR + bias",
        "MSE": result_3b["mse"],
        "Next prediction": result_3b["next_prediction"]
    }
])

display(lms_summary.sort_values("MSE"))

In [ ]:
# Plot actual values and online one-step predictions.

plt.figure(figsize=(11, 5))

plt.plot(
    np.arange(len(close)),
    close,
    marker="o",
    linewidth=2,
    label="Actual Close/Last"
)

for result, label in [
    (result_2, "2nd FIR"),
    (result_2b, "2nd FIR + bias"),
    (result_3, "3rd FIR"),
    (result_3b, "3rd FIR + bias")
]:
    plt.plot(
        result["sample_indices"],
        result["predictions"],
        marker=".",
        alpha=0.8,
        label=label
    )

plt.xlabel("Chronological sample index")
plt.ylabel("Price")
plt.title("Adaptive FIR One-Step Predictions")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

# Question 2(e) — Logistic / Softmax Regression

## Why a modeling definition is necessary

Ordinary LMS predicts a real-valued price directly.

Logistic regression, however, predicts **classes**.

To remain consistent with the homework's Softmax learning objective, define the next-day change

$$
\Delta[n]
=
x[n]-x[n-1].
$$

Using only the training period, split the changes into three approximately balanced classes:

$$
\boxed{
\text{Down},\quad
\text{Middle},\quad
\text{Up}.
}
$$

The class boundaries are the 1/3 and 2/3 quantiles of the training changes.

For each observation, use the previous three close prices as features:

$$
\mathbf z[n]
=
\begin{bmatrix}
x[n-1]\\
x[n-2]\\
x[n-3]
\end{bmatrix}.
$$

Multinomial logistic regression produces Softmax probabilities

$$
P(C=c\mid\mathbf z).
$$

To obtain a numerical next-price prediction, calculate the average price change for each class from the training data:

$$
\bar{\Delta}_c.
$$

Then use the Softmax expected change

$$
\boxed{
\widehat{\Delta}
=
\sum_c
P(C=c\mid\mathbf z)
\bar{\Delta}_c
}
$$

and finally

$$
\boxed{
\hat{x}[n]
=
x[n-1]
+
\widehat{\Delta}.
}
$$

This gives a real-valued estimate that can be compared with the LMS predictors using MSE.

## Use a chronological holdout for a fair comparison

To avoid evaluating a model on the same future observations used to fit it:

- first 15 prices: training;
- last 7 prices: testing.

The same seven test prices will later be used to compare **all** predictors fairly.

In [ ]:
SOFTMAX_ORDER = 3
TRAIN_END = 15  # indices 0,...,14 train; 15,...,21 test

features = []
changes = []
targets = []
indices = []

for n in range(SOFTMAX_ORDER, len(close)):
    features.append([
        close[n-1],
        close[n-2],
        close[n-3]
    ])
    changes.append(close[n] - close[n-1])
    targets.append(close[n])
    indices.append(n)

features = np.asarray(features, dtype=float)
changes = np.asarray(changes, dtype=float)
targets = np.asarray(targets, dtype=float)
indices = np.asarray(indices)

train_mask = indices < TRAIN_END
test_mask = indices >= TRAIN_END

train_changes = changes[train_mask]

lower_cut, upper_cut = np.quantile(
    train_changes,
    [1/3, 2/3]
)

def movement_class(delta, lower, upper):
    delta = np.asarray(delta)
    return np.where(
        delta < lower,
        0,
        np.where(delta <= upper, 1, 2)
    )

classes = movement_class(
    changes,
    lower_cut,
    upper_cut
)

train_classes = classes[train_mask]

class_names = np.array(["Down", "Middle", "Up"])

class_mean_change = np.array([
    train_changes[train_classes == c].mean()
    for c in range(3)
])

print("Training lower cut =", lower_cut)
print("Training upper cut =", upper_cut)
print("Training class counts =", np.bincount(train_classes))
print("Mean change per class =", dict(
    zip(class_names, class_mean_change)
))

In [ ]:
softmax_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=5000,
        random_state=0
    )
)

softmax_model.fit(
    features[train_mask],
    train_classes
)

test_probabilities = softmax_model.predict_proba(
    features[test_mask]
)

test_last_price = features[test_mask][:, 0]

expected_change = (
    test_probabilities @ class_mean_change
)

softmax_price_predictions = (
    test_last_price + expected_change
)

softmax_test_targets = targets[test_mask]

softmax_mse = float(np.mean(
    (softmax_price_predictions - softmax_test_targets)**2
))

softmax_table = pd.DataFrame({
    "Index": indices[test_mask],
    "True price": softmax_test_targets,
    "Predicted price": softmax_price_predictions,
    "P(Down)": test_probabilities[:, 0],
    "P(Middle)": test_probabilities[:, 1],
    "P(Up)": test_probabilities[:, 2]
})

display(softmax_table)

print("Softmax holdout MSE =", softmax_mse)

For the fallback dataset, the Softmax-based price reconstruction gives the common holdout MSE

$$
\boxed{
\mathrm{MSE}_{\text{Softmax}}
\approx
1.433888.
}
$$

In [ ]:
# Fit the Softmax predictor on all available observations
# to estimate the next unknown value after the final date.

all_lower, all_upper = np.quantile(
    changes,
    [1/3, 2/3]
)

all_classes = movement_class(
    changes,
    all_lower,
    all_upper
)

all_class_mean_change = np.array([
    changes[all_classes == c].mean()
    for c in range(3)
])

softmax_all = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=5000,
        random_state=0
    )
)

softmax_all.fit(features, all_classes)

next_features = np.array([[
    close[-1],
    close[-2],
    close[-3]
]])

next_probabilities = softmax_all.predict_proba(
    next_features
)[0]

next_expected_change = float(
    next_probabilities @ all_class_mean_change
)

softmax_next_price = float(
    close[-1] + next_expected_change
)

print("Next-class probabilities:")
for name, prob in zip(class_names, next_probabilities):
    print(f"  {name:6s}: {prob:.6f}")

print("Expected next change =", next_expected_change)
print("Softmax next-price estimate =", softmax_next_price)

Using all available fallback observations, the Softmax next-price estimate is approximately

$$
\boxed{
49.00398.
}
$$

The exact number will update automatically if you upload a different professor-provided CSV.

# Question 2(f) — Which Predictor Is Best?

The online LMS MSEs in parts (a)–(d) measure prediction while the filter is still adapting.

The Softmax model was evaluated on a separate holdout period.

For a **fair model comparison**, evaluate all five predictors on the same final seven observations.

For each LMS model:

1. adapt only on the first 15 observations;
2. freeze the learned coefficients;
3. predict the last seven prices using their actual past values;
4. calculate MSE on exactly those seven prices.

This gives the same test set for all models.

In [ ]:
def train_nlms_then_test(
    series,
    order,
    use_bias=False,
    train_end=15,
    mu=1.0,
    bias_init=0.0
):
    series = np.asarray(series, dtype=float)

    if use_bias:
        w = np.concatenate([
            np.full(order, 1.0/order),
            [bias_init]
        ])
    else:
        w = np.full(order, 1.0/order)

    # Adapt only on training period.
    for n in range(order, train_end):
        u = np.array([
            series[n-k]
            for k in range(1, order+1)
        ], dtype=float)

        if use_bias:
            u = np.concatenate([u, [1.0]])

        prediction = float(w @ u)
        error = float(series[n] - prediction)

        w = w + mu * error / float(u @ u) * u

    # Freeze weights and evaluate common test period.
    predictions = []
    true_values = []

    for n in range(train_end, len(series)):
        u = np.array([
            series[n-k]
            for k in range(1, order+1)
        ], dtype=float)

        if use_bias:
            u = np.concatenate([u, [1.0]])

        predictions.append(float(w @ u))
        true_values.append(float(series[n]))

    predictions = np.asarray(predictions)
    true_values = np.asarray(true_values)

    mse = float(np.mean(
        (predictions - true_values)**2
    ))

    return {
        "weights": w,
        "predictions": predictions,
        "targets": true_values,
        "mse": mse
    }

test_2 = train_nlms_then_test(
    close, 2, False, TRAIN_END
)

test_2b = train_nlms_then_test(
    close, 2, True, TRAIN_END
)

test_3 = train_nlms_then_test(
    close, 3, False, TRAIN_END
)

test_3b = train_nlms_then_test(
    close, 3, True, TRAIN_END
)

fair_comparison = pd.DataFrame([
    {
        "Predictor": "2nd-order LMS",
        "Common test MSE": test_2["mse"]
    },
    {
        "Predictor": "2nd-order LMS + bias",
        "Common test MSE": test_2b["mse"]
    },
    {
        "Predictor": "3rd-order LMS",
        "Common test MSE": test_3["mse"]
    },
    {
        "Predictor": "3rd-order LMS + bias",
        "Common test MSE": test_3b["mse"]
    },
    {
        "Predictor": "Softmax price reconstruction",
        "Common test MSE": softmax_mse
    }
]).sort_values("Common test MSE")

display(fair_comparison)

For the fallback Homework #3 data, the common seven-point holdout MSEs are approximately:

| Predictor | Common test MSE |
|---|---:|
| Softmax price reconstruction | $1.433888$ |
| 3rd-order LMS | $2.438903$ |
| 3rd-order LMS + bias | $2.438903$ |
| 2nd-order LMS | $3.238148$ |
| 2nd-order LMS + bias | $3.238154$ |

Therefore, under this **common held-out test evaluation**,

$$
\boxed{
\text{Softmax price reconstruction has the lowest MSE.}
}
$$

Among the pure LMS predictors,

$$
\boxed{
\text{the 3rd-order LMS without bias is the best.}
}
$$

The bias terms do not materially improve this short dataset because their learned values are very close to zero.

In [ ]:
plt.figure(figsize=(9, 4.8))

plot_df = fair_comparison.sort_values(
    "Common test MSE",
    ascending=False
)

plt.barh(
    plot_df["Predictor"],
    plot_df["Common test MSE"]
)

plt.xlabel("MSE on common seven-point test set")
plt.title("Question 2(f) — Fair Predictor Comparison")
plt.grid(axis="x", alpha=0.3)
plt.show()

# Clean Source Code for Question 2

The professor says the Question 2 source code should be attached at the end of the written Question 1 solution in a **single PDF**.

The notebook cells above are the explained version. The following cell is a compact, submission-ready version of the core code.

In [ ]:
# ==============================================================
# MENG 417 Homework #3 — Question 2
# Adaptive LMS and Softmax comparison
# ==============================================================

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline


def adaptive_nlms(series, order, use_bias=False):
    series = np.asarray(series, dtype=float)

    if use_bias:
        w = np.r_[np.ones(order)/order, 0.0]
    else:
        w = np.ones(order)/order

    errors = []

    for n in range(order, len(series)):
        u = np.array(
            [series[n-k] for k in range(1, order+1)]
        )

        if use_bias:
            u = np.r_[u, 1.0]

        prediction = w @ u
        error = series[n] - prediction
        errors.append(error)

        w = w + error/(u @ u)*u

    u_next = np.array(
        [series[-k] for k in range(1, order+1)]
    )

    if use_bias:
        u_next = np.r_[u_next, 1.0]

    next_prediction = w @ u_next
    mse = np.mean(np.square(errors))

    return mse, next_prediction, w


# 'close' must be chronological: oldest -> newest.
# Example:
# data = pd.read_csv("HistoricalQuotes.csv")
# close = (
#     data["Close/Last"]
#     .str.replace("$", "", regex=False)
#     .astype(float)
#     .to_numpy()[::-1]
# )

mse2, next2, w2 = adaptive_nlms(close, 2, False)
mse2b, next2b, w2b = adaptive_nlms(close, 2, True)
mse3, next3, w3 = adaptive_nlms(close, 3, False)
mse3b, next3b, w3b = adaptive_nlms(close, 3, True)

print("2nd FIR MSE:", mse2, "Next:", next2)
print("2nd FIR + bias MSE:", mse2b, "Next:", next2b)
print("3rd FIR MSE:", mse3, "Next:", next3)
print("3rd FIR + bias MSE:", mse3b, "Next:", next3b)

# Final Answer Summary

## Question 1

Using the 2nd-order normalized LMS predictor initialized with

$$
\mathbf w^{(0)}
=
[0.5,\;0.5]^T,
$$

the final coefficients are approximately

$$
\boxed{
\mathbf w
=
[0.80115785,\;0.58688801]^T.
}
$$

Therefore,

$$
\boxed{
\hat{x}[5]
\approx
6378.95.
}
$$

---

## Question 2(a)

2nd-order adaptive FIR:

$$
\boxed{
\mathrm{MSE}\approx1.987393
}
$$

$$
\boxed{
\hat{x}_{\text{next}}\approx48.872017.
}
$$

---

## Question 2(b)

2nd-order adaptive FIR with bias:

$$
\boxed{
\mathrm{MSE}\approx1.987395
}
$$

$$
\boxed{
\hat{x}_{\text{next}}\approx48.872016.
}
$$

---

## Question 2(c)

3rd-order adaptive FIR:

$$
\boxed{
\mathrm{MSE}\approx1.715399
}
$$

$$
\boxed{
\hat{x}_{\text{next}}\approx49.550783.
}
$$

---

## Question 2(d)

3rd-order adaptive FIR with bias:

$$
\boxed{
\mathrm{MSE}\approx1.715404
}
$$

$$
\boxed{
\hat{x}_{\text{next}}\approx49.550783.
}
$$

---

## Question 2(e)

A 3-class Softmax model is used for next-price movement and converted to a numerical price estimate by probability-weighted mean movement.

Using all available fallback observations:

$$
\boxed{
\hat{x}_{\text{next, Softmax}}
\approx49.00398.
}
$$

On the common seven-point holdout set:

$$
\boxed{
\mathrm{MSE}_{\text{Softmax}}
\approx1.433888.
}
$$

---

## Question 2(f)

On the **same held-out observations** for every model:

$$
\boxed{
\text{Softmax price reconstruction has the lowest MSE.}
}
$$

Among the LMS-only predictors:

$$
\boxed{
\text{3rd-order LMS without bias is the best.}
}
$$

# MENG 417 Homework #3 — Mature Cheat Sheet

## 1. FIR prediction

An $M$th-order FIR predictor uses the last $M$ observations:

$$
\boxed{
\hat{x}[n]
=
\sum_{k=1}^{M}
w_kx[n-k].
}
$$

---

## 2. 2nd-order predictor

$$
\boxed{
\hat{x}[n]
=
w_1x[n-1]
+
w_2x[n-2].
}
$$

Input vector:

$$
\mathbf u[n]
=
[x[n-1],x[n-2]]^T.
$$

---

## 3. 3rd-order predictor

$$
\boxed{
\hat{x}[n]
=
w_1x[n-1]
+
w_2x[n-2]
+
w_3x[n-3].
}
$$

---

## 4. Bias

A biased predictor adds a constant parameter:

$$
\boxed{
\hat{x}[n]
=
\mathbf w^T\mathbf u[n]+b.
}
$$

Using an augmented vector:

$$
\tilde{\mathbf u}
=
[\mathbf u^T,1]^T.
$$

---

## 5. Prediction error

$$
\boxed{
e[n]
=
x[n]-\hat{x}[n].
}
$$

The sign tells us the direction of the prediction error.

---

## 6. Mean squared error

The professor defines

$$
\boxed{
\mathrm{MSE}
=
\frac1N
\sum
(\hat{x}[n]-x[n])^2.
}
$$

Smaller MSE means better numerical prediction.

Do **not** replace MSE with mean absolute error.

---

## 7. LMS

Basic LMS has the form

$$
\mathbf w[n+1]
=
\mathbf w[n]
+
\mu e[n]\mathbf u[n].
$$

The parameter

$$
\mu
$$

is the learning rate.

---

## 8. Normalized LMS

Normalized LMS scales the update by the input energy:

$$
\boxed{
\mathbf w[n+1]
=
\mathbf w[n]
+
\frac{\mu e[n]}
{\|\mathbf u[n]\|^2}
\mathbf u[n].
}
$$

This reduces sensitivity to the magnitude of the input.

This homework's course-aligned calculation uses

$$
\mu=1.
$$

---

## 9. Online prediction

For adaptive prediction, the correct order is:

1. form the input from past samples;
2. predict the current sample;
3. calculate error against the current true sample;
4. record the squared error;
5. update the weights.

Never use the current/future true value inside the input vector.

---

## 10. Chronological order matters

Historical quote files are often stored newest first.

Prediction must use

$$
\boxed{
\text{oldest}\rightarrow\text{newest}.
}
$$

Otherwise the model would accidentally use the future to predict the past.

---

## 11. Logistic regression and Softmax

Binary logistic regression uses the sigmoid.

Multiclass logistic regression uses Softmax:

$$
\boxed{
P(C=c\mid\mathbf z)
=
\frac{
e^{s_c}
}{
\sum_j e^{s_j}
}.
}
$$

The probabilities satisfy

$$
\sum_cP(C=c\mid\mathbf z)=1.
$$

---

## 12. Why Softmax cannot directly predict a raw stock price

A raw price such as

$$
49.35
$$

is continuous.

Softmax predicts a class.

Therefore the continuous target must first be represented by meaningful classes, or the model must be replaced by a true regression algorithm.

In this notebook:

$$
\text{price movement}
\rightarrow
\{\text{Down, Middle, Up}\}.
$$

Then Softmax probabilities are converted back to a numerical change using an expected-value calculation.

---

## 13. Fair model comparison

To compare models fairly:

$$
\boxed{
\text{use the same test observations for every model}.
}
$$

Comparing training error from one model with test error from another model is not fair.

---

# Common Mistakes to Avoid

1. Do not calculate MSE with the wrong target index.
2. Do not use future values as predictor inputs.
3. Do not forget to reverse a newest-first financial CSV into chronological order.
4. Do not call mean absolute error "MSE."
5. Do not treat logistic regression as ordinary continuous regression.
6. Do not compare models on different evaluation sets and call one "best."
7. Do not hide initialization choices; state them.
8. Always verify the final next-value prediction using the final learned coefficients.

---

# Final Mental Model

$$
\boxed{
\text{past samples}
\rightarrow
\text{FIR predictor}
\rightarrow
\hat{x}[n]
\rightarrow
e[n]
\rightarrow
\text{LMS weight update}.
}
$$

For Softmax:

$$
\boxed{
\text{past prices}
\rightarrow
\text{movement class probabilities}
\rightarrow
\text{expected change}
\rightarrow
\text{predicted next price}.
}
$$

# Final Numerical Validation

This cell checks the fixed numerical answers produced from the fallback Homework #3 data.

In [ ]:
# Q1
assert abs(x5_prediction - 6378.950116584337) < 1e-9

# Assignment-style adaptive LMS MSE
assert abs(result_2["mse"] - 1.987392631984815) < 1e-10
assert abs(result_2b["mse"] - 1.9873952795124097) < 1e-10
assert abs(result_3["mse"] - 1.7153993827654448) < 1e-10
assert abs(result_3b["mse"] - 1.715403607703585) < 1e-10

# Next predictions
assert abs(result_2["next_prediction"] - 48.87201671317645) < 1e-10
assert abs(result_3["next_prediction"] - 49.55078298308418) < 1e-10

# Softmax common-test result
assert abs(softmax_mse - 1.4338877227395643) < 1e-9

print("All Homework #3 fixed numerical checks passed.")